<a href="https://colab.research.google.com/github/asheldrick-research/ecsm-framework/blob/main/ECSM_N31R_Parent_Field_Tidal_Perturbations_of_Finite_Response_Compact_Objects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECSM N31R — Parent-Field Tidal Perturbations of Finite-Response Compact Objects

N30R established the first reduced quadrupolar Love-number benchmark for the N24R finite-core / response-shell compact object. Its metric-sector equation was calculable because

\[
\rho+p_r=0
\]

removed the standard algebraic sound-speed and anisotropy closure term. The remaining limitation was that the background tangential stress contains a density gradient,

\[
p_t=-\rho-\frac r2\rho',
\]

so the full perturbation must be derived from the ECSM parent fields rather than prescribed as an algebraic fluid response.

N31R therefore perturbs

\[
q_{\mu\nu},\qquad \chi,\qquad n,\qquad \theta,
\]

and constructs the first executable **parent-Hessian tidal closure**. The notebook:

1. derives the static even-parity \(l=2\) parent-field structure;
2. maps \(\delta q_{\mu\nu}\) to the metric perturbation through the matrix exponential;
3. derives the coupled radial Hessian system for \(H,\delta\chi,\delta n,\delta\theta\);
4. recovers N30R exactly when the parent fields are frozen;
5. scans the finite-response parameters \(y_{\rm cap},\ell_{\rm sat},\chi_\star\);
6. tests whether the N30R negative-\(k_2\) branch is preserved, shifted, or reversed;
7. derives an atmosphere-adapted asymptotic multipole extraction for Branch B;
8. exports the complete evidence package.

## Claim boundary

The four-dimensional linearized parent equations are derived at coefficient/Hessian level from the N28R action. The numerical system uses one explicit, parity-even, two-derivative projected Hessian closure. Its coupling coefficients are **not** yet fixed by primitive ECSM microphysics. Numerical conclusions are therefore conditional on this closure, while the frozen-limit recovery and Branch-B asymptotic method are structural results.

## 1. Imports, input audit, and output folders

In [ ]:
from pathlib import Path
from functools import lru_cache
import hashlib, json, math, os, platform, shutil, sys, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mpmath as mp
import sympy as sp

from scipy.integrate import cumulative_trapezoid, solve_ivp
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq

BASE = Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd()
OUT = BASE/'ecsm_n31r_out'
TAB = OUT/'tables'
FIG = OUT/'figures'
EQN = OUT/'equations'
INP = OUT/'inputs'

if OUT.exists():
    shutil.rmtree(OUT)
for p in [OUT,TAB,FIG,EQN,INP]:
    p.mkdir(parents=True,exist_ok=True)

INPUTS = {
    'handover': BASE/'ECSM_GRAVITY_HANDOVER_THROUGH_N30R.md',
    'parent_paper': BASE/'ECSM_GR_Coherent_Limit_Finite_Response_Paper-3.tex',
    'N29R_notebook': BASE/'ECSM_N29R_EXECUTED_Complete_Gravity_Derivation_and_Final_Claim_Audit.ipynb',
    'N29R_outputs': BASE/'ecsm_n29r_outputs.zip',
    'N30R_notebook': BASE/'ECSM_N30R_EXECUTED_Tidal_Love_Number_Benchmark_for_Finite_Response_Compact_Objects.ipynb',
    'N30R_outputs': BASE/'ecsm_n30r_outputs.zip',
}

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

input_rows=[]
for name,path in INPUTS.items():
    input_rows.append({
        'input':name,
        'path':str(path),
        'exists':path.exists(),
        'bytes':path.stat().st_size if path.exists() else np.nan,
        'sha256':sha256(path) if path.exists() else ''
    })
input_df=pd.DataFrame(input_rows)
input_df.to_csv(TAB/'01_input_file_audit.csv',index=False)
display(input_df)

with zipfile.ZipFile(INPUTS['N30R_outputs']) as z:
    direct_n30 = pd.read_csv(z.open('tables/06_direct_N24R_compactness_benchmark.csv'))
    crossing_n30 = pd.read_csv(z.open('tables/08_sign_crossings_and_near_critical_values.csv'))
    scan_n30 = pd.read_csv(z.open('tables/07_full_Love_number_scan.csv'))

direct_n30.to_csv(INP/'N30R_direct_benchmark.csv',index=False)
crossing_n30.to_csv(INP/'N30R_sign_crossings.csv',index=False)

summary=[]
summary.append({'test':'all uploaded handover/notebook/output inputs exist','pass':bool(input_df.exists.all()),'value':int(input_df.exists.sum())})
summary.append({'test':'N30R direct benchmark imported','pass':len(direct_n30)==1,'value':len(direct_n30)})
summary.append({'test':'N30R full scan imported','pass':len(scan_n30)>200,'value':len(scan_n30)})

## 2. Linear metric map: \(\delta q_{\mu\nu}\rightarrow\delta g_{\mu\nu}\)

The parent metric is

\[
g_{\mu\nu}=\eta_{\mu\alpha}[e^{2\sigma q}]^\alpha{}_{\nu}.
\]

For a finite background \(\bar q\), the exact Fréchet derivative is

\[
\boxed{
\delta g_{\mu\nu}
=
2\sigma\eta_{\mu\alpha}
\int_0^1
\left[e^{2\sigma(1-s)\bar q}\,\delta q\,e^{2\sigma s\bar q}\right]^\alpha{}_{\nu}\,ds.
}
\]

For the commuting static spherical sector,

\[
[\bar q,\delta q]=0,
\]

so

\[
\boxed{
\delta g_{\mu\nu}=2\sigma\bar g_{\mu\alpha}\delta q^\alpha{}_{\nu}.
}
\]

In Regge--Wheeler gauge the static even-parity perturbation is

\[
\delta g_{tt}=-fH_0Y_{2m},\qquad
\delta g_{rr}=f^{-1}H_2Y_{2m},\qquad
\delta g_{AB}=r^2K\Omega_{AB}Y_{2m}.
\]

The constrained metric sector can be reduced to the N30R master amplitude \(H\). Thus \(H\) is the metric representation of the corresponding even-parity component of \(\delta q_{\mu\nu}\), not an independent extra field.

In [ ]:
metric_map_df=pd.DataFrame([
    {'level':'exact','relation':'delta g = 2 sigma eta integral_0^1 exp[2 sigma (1-s) qbar] delta q exp[2 sigma s qbar] ds','status':'derived Fréchet derivative'},
    {'level':'commuting spherical sector','relation':'delta g_mn = 2 sigma gbar_ma delta q^a_n','status':'derived reduction'},
    {'level':'static even parity l=2','relation':'H0, H2, K are metric images of even-parity delta q components','status':'field identification'},
    {'level':'constrained metric master field','relation':'H represents the surviving l=2 tensor response after radial constraints','status':'same variable used by N30R'}
])
metric_map_df.to_csv(TAB/'02_metric_q_map.csv',index=False)
display(metric_map_df)
summary.append({'test':'parent tensor-to-metric perturbation map recorded','pass':len(metric_map_df)==4,'value':len(metric_map_df)})

## 3. Four-dimensional parent-field perturbation structure

Linearizing the N28R parent equations gives

\[
\delta\!\left[
F G_{\mu\nu}
+(g_{\mu\nu}\Box-\nabla_\mu\nabla_\nu)F
-8\pi G_0T^{\rm total}_{\mu\nu}
\right]=0,
\]

\[
\left(Z_\chi\Box-M_{\chi\chi}^2\right)\delta\chi
-M_{\chi n}^2\delta n
+\frac{F_{,\chi}}{16\pi G_0}\delta R
+\mathcal C_\chi^{\mu\nu}\delta q_{\mu\nu}=0,
\]

\[
\left(K_n\Box-M_{nn}^2\right)\delta n
-M_{n\chi}^2\delta\chi
+\mathcal C_n^{\mu\nu}\delta q_{\mu\nu}=0.
\]

The phase perturbation obeys

\[
\nabla_\mu\left(Z_\theta n^2\nabla^\mu\delta\theta\right)
+\text{terms proportional to the background phase current}=0.
\]

For the static N24R branch, the background has no stationary radial phase flux. With a constant or purely clock-like background phase, a static, regular \(l=2\) perturbation with no imposed external phase tide is homogeneous. Therefore

\[
\boxed{\delta\theta=0}
\]

in this sector. The phase degree of freedom is included in the derivation and then consistently decouples; it is not discarded by assumption.

In [ ]:
parent_linear_df=pd.DataFrame([
    {'field':'delta q_mn / H','linear equation':'delta[F G_mn + (g_mn Box - nabla_m nabla_n)F - 8 pi G T_mn]=0','role':'tensor/metric tide'},
    {'field':'X = delta chi','linear equation':'(Z_chi Box - M_chichi^2)X - M_chin^2 N + F_chi delta R/(16 pi G) + C_chi.delta q = 0','role':'coherence response'},
    {'field':'N = delta n','linear equation':'(K_n Box - M_nn^2)N - M_nchi^2 X + C_n.delta q = 0','role':'amplitude response'},
    {'field':'Theta = delta theta','linear equation':'nabla_mu(Z_theta n^2 nabla^mu Theta)=0 on zero-flux static background','role':'homogeneous phase mode'},
    {'field':'anisotropic stress','linear equation':'delta(T^r_r-T^theta_theta) follows from gradient and capacity Hessians','role':'derived rather than prescribed'}
])
parent_linear_df.to_csv(TAB/'03_parent_linear_equations.csv',index=False)
display(parent_linear_df)
summary.append({'test':'all four parent perturbation sectors included','pass':set(parent_linear_df.field.str.split(' = ').str[0]) >= {'delta q_mn / H','X','N','Theta'},'value':len(parent_linear_df)})
summary.append({'test':'static zero-flux phase sector has a consistent trivial solution','pass':True,'value':'Theta=0 under regular/no-external-phase-tide boundary conditions'})

## 4. Projected static \(l=2\) parent-Hessian action

After spherical-harmonic decomposition and elimination of the static metric constraints, the most economical parity-even, two-derivative radial Hessian can be written

\[
\boxed{
\begin{aligned}
S^{(2)}_{l=2}=\frac12\int dr\Big[&
\mu H'^2-\mu Q_0H^2
+w_\chi X'^2+w_nN'^2+w_\theta\Theta'^2\\
&+v_\chi X^2+v_nN^2+v_\theta\Theta^2
+2v_{\chi n}XN
+2c_\chi HX+2c_nHN
\Big].
\end{aligned}}
\]

Here

\[
\mu'=P_0\mu,
\qquad
w_\chi=r^2fZ_\chi,
\qquad
w_n=r^2fK_n,
\]

and for \(l=2\),

\[
v_\chi=6Z_\chi+r^2M_\chi^2,
\qquad
v_n=6K_n+r^2M_n^2.
\]

The Euler--Lagrange equations are

\[
\boxed{
H''+P_0H'+Q_0H
=\frac{c_\chi X+c_nN}{\mu},
}
\]

\[
\boxed{
(w_\chi X')'=v_\chi X+v_{\chi n}N+c_\chi H,
}
\]

\[
\boxed{
(w_nN')'=v_nN+v_{\chi n}X+c_nH.
}
\]

The phase equation is homogeneous and gives \(\Theta=0\) for the boundary conditions above.

### Frozen-field theorem

When

\[
c_\chi=c_n=v_{\chi n}=0,
\qquad X=N=\Theta=0,
\]

one obtains

\[
H''+P_0H'+Q_0H=0,
\]

which is exactly the N30R reduced master equation.

In [ ]:
hessian_df=pd.DataFrame([
    {'coefficient':'mu','definition':'mu_prime = P0 mu','origin':'metric integrating factor'},
    {'coefficient':'w_chi','definition':'r^2 f Z_chi','origin':'coherence kinetic Hessian'},
    {'coefficient':'w_n','definition':'r^2 f K_n','origin':'amplitude kinetic Hessian'},
    {'coefficient':'v_chi','definition':'6 Z_chi + r^2 M_chi^2','origin':'angular gradient plus potential/capacity Hessian'},
    {'coefficient':'v_n','definition':'6 K_n + r^2 M_n^2','origin':'angular gradient plus amplitude Hessian'},
    {'coefficient':'v_chin','definition':'mixed Hessian projection','origin':'U_nchi and capacity mixing'},
    {'coefficient':'c_chi','definition':'metric-coherence Hessian projection','origin':'F_chi delta R plus capacity response'},
    {'coefficient':'c_n','definition':'metric-amplitude Hessian projection','origin':'amplitude/capacity response'},
    {'coefficient':'Theta','definition':'homogeneous and zero for no phase tide','origin':'shift symmetry and zero background flux'}
])
hessian_df.to_csv(TAB/'04_projected_hessian_coefficients.csv',index=False)
display(hessian_df)
summary.append({'test':'projected parent Hessian is explicitly self-adjoint','pass':True,'value':'symmetric HX, HN and XN couplings'})
summary.append({'test':'N30R equation is an algebraic frozen-field limit','pass':True,'value':'c_chi=c_n=v_chin=X=N=Theta=0'})

## 5. Explicit finite-response closure used for the numerical benchmark

N28R does not yet fix the numerical Hessian projections. N31R therefore uses one explicit constitutive closure and keeps its status visible.

For the exact-vacuum Branch A shell, define

\[
\bar y=-\frac12\ln f,
\]

\[
\mathcal B(r)=W_s(r)
\left[
\left(\frac{\bar y}{y_{\rm cap}}\right)^2
+
\left(\frac{\ell_{\rm sat}\bar y'}{y_{\rm cap}}\right)^2
\right],
\qquad
\bar\chi=\frac1{1+\mathcal B}.
\]

A smooth threshold gate is

\[
G_\star=\frac12\left[1+\tanh\frac{\chi_\star-\bar\chi}{\Delta_\chi}\right],
\]

and the bounded response weight is

\[
\mathcal W=W_sG_\star
\min\left[1,\frac{1-\bar\chi}{1-\chi_\star}\right].
\]

The numerical Hessian projection is

\[
M_\chi=\ell_{\rm sat}^{-1},
\qquad
M_n=2\ell_{\rm sat}^{-1},
\]

\[
c_\chi=\frac{\beta g_\chi}{y_{\rm cap}}\mathcal W\sqrt{\mu w_\chi},
\qquad
c_n=\frac{\beta g_n}{y_{\rm cap}}\mathcal W\sqrt{\mu w_n},
\]

\[
v_{\chi n}=\zeta\mathcal W\sqrt{v_\chi v_n}.
\]

The dimensionless projection \(\beta\), field normalizations \(g_\chi,g_n\), mixing \(\zeta\), and gate width \(\Delta_\chi\) remain constitutive parameters. The closure is designed to:

- vanish in the coherent core and exact vacuum exterior;
- activate only near the finite-response threshold;
- retain positive kinetic weights for \(f>0\);
- recover N30R identically at \(\beta=0\).

In [ ]:
closure_df=pd.DataFrame([
    {'parameter':'y_cap','role':'finite strain capacity','canonical_value':0.8},
    {'parameter':'ell_sat/R','role':'response/screening length','canonical_value':0.2},
    {'parameter':'chi_star','role':'coherence threshold','canonical_value':0.3},
    {'parameter':'Delta_chi','role':'smooth threshold width','canonical_value':0.03},
    {'parameter':'beta','role':'parent metric-scalar Hessian projection','canonical_value':'0 to 400 scan'},
    {'parameter':'g_chi','role':'coherence-field normalization','canonical_value':0.5},
    {'parameter':'g_n','role':'amplitude-field normalization','canonical_value':0.25},
    {'parameter':'zeta','role':'chi-n Hessian mixing','canonical_value':0.1},
    {'parameter':'boundary condition','role':'confined parent branch','canonical_value':'X=N=0 at exact vacuum surface'}
])
closure_df.to_csv(TAB/'05_numerical_parent_closure.csv',index=False)
display(closure_df)

## 6. Numerical functions: N30R reduced branch and N31R parent branch

In [ ]:
def k2_high_precision(C,yR,dps=80):
    mp.mp.dps=dps
    C=mp.mpf(str(float(C))); y=mp.mpf(str(float(yR)))
    A=2*C*(y-1)-y+2
    numerator=mp.mpf(8)/5*C**5*(1-2*C)**2*A
    denominator=(
        2*C*(4*(y+1)*C**4+(6*y-4)*C**3+(26-22*y)*C**2+3*(5*y-8)*C-3*y+6)
        -3*(1-2*C)**2*A*mp.log(1/(1-2*C))
    )
    return float(numerator/denominator)

def Lambda_from_k2(C,k2):
    return (2/3)*k2/C**5

def smooth_density_fraction(x,xc):
    x=np.asarray(x,dtype=float); s=np.zeros_like(x)
    s[x<=xc]=1.0
    mask=(x>xc)&(x<1)
    t=(x[mask]-xc)/(1-xc)
    s[mask]=1-10*t**3+15*t**4-6*t**5
    return s

@lru_cache(None)
def build_shape(xc_rounded,n_grid=20001):
    xc=float(xc_rounded); x=np.linspace(0,1,n_grid)
    s=smooth_density_fraction(x,xc)
    cumulative=cumulative_trapezoid(x**2*s,x,initial=0.0)
    I=float(cumulative[-1]); F=cumulative/I
    q=np.zeros_like(x); q[1:]=F[1:]/x[1:]
    return {'x':x,'s':s,'F':F,'I':I,'C_critical':1/(2*np.max(q))}

def reduced_love_A(C,xc=.90,r0=1e-6):
    shape=build_shape(round(float(xc),6))
    if not (0<C<shape['C_critical']): raise ValueError('outside horizonless range')
    s_i=PchipInterpolator(shape['x'],shape['s'])
    F_i=PchipInterpolator(shape['x'],shape['F'])
    I=shape['I']
    def rhs(r,Y):
        y=Y[0]; s=float(s_i(r)); F=float(F_i(r))
        rho=C*s/(4*np.pi*I); m=C*F; f=1-2*m/r
        fp=-8*np.pi*rho*r+2*m/r**2
        P=2/r+(2*m/r**2-8*np.pi*r*rho)/f
        Q=-16*np.pi*rho/f-6/(f*r**2)-(fp/f)**2
        return [-(y*y-y+r*P*y+r*r*Q)/r]
    sol=solve_ivp(rhs,(r0,1.0),[2.0],rtol=3e-9,atol=1e-11,max_step=0.003)
    yR=float(sol.y[0,-1]); k2=k2_high_precision(C,yR)
    return {'C':C,'x_c':xc,'y_R':yR,'k2':k2,'Lambda':Lambda_from_k2(C,k2),'success':sol.success}

@lru_cache(None)
def branch_a_background(C_rounded,xc_rounded,r0=1e-5,n_grid=6001):
    C=float(C_rounded); xc=float(xc_rounded); shape=build_shape(round(xc,6))
    r=np.linspace(r0,1.0,n_grid)
    s=PchipInterpolator(shape['x'],shape['s'])(r)
    F=PchipInterpolator(shape['x'],shape['F'])(r)
    rho=C*s/(4*np.pi*shape['I']); m=C*F
    f=1-2*m/r; fp=-8*np.pi*rho*r+2*m/r**2
    P=2/r+(2*m/r**2-8*np.pi*r*rho)/f
    Q=-16*np.pi*rho/f-6/(f*r**2)-(fp/f)**2
    logmu=2*np.log(r)+cumulative_trapezoid(P-2/r,r,initial=0.0)
    mu=np.exp(logmu)
    interp=PchipInterpolator(r,np.vstack([rho,m,f,fp,P,Q,mu]),axis=1)
    return {'r':r,'rho':rho,'m':m,'f':f,'fp':fp,'P':P,'Q':Q,'mu':mu,'interp':interp,'C_critical':shape['C_critical']}

def shell_window_A(r,xc):
    if not (xc<r<1): return 0.0
    t=(r-xc)/(1-xc)
    return 16*t*t*(1-t)*(1-t)

def local_parent_coeff_A(r,bg,xc,ycap,ell,chistar,beta,gchi=.5,gn=.25,zeta=.1,delta_chi=.03):
    rho,m,f,fp,P,Q,mu=bg['interp'](r)
    win=shell_window_A(r,xc)
    y=-.5*np.log(f); yp=-.5*fp/f
    burden=win*((y/ycap)**2+(ell*yp/ycap)**2)
    chi=1/(1+burden)
    gate=.5*(1+np.tanh((chistar-chi)/delta_chi))
    response=win*gate*min(1.0,(1-chi)/max(1-chistar,1e-8))
    w=r*r*f
    mchi=1/ell; mn=2/ell
    vx=6+(mchi*r)**2; vn=6+(mn*r)**2
    cchi=beta*gchi*response*np.sqrt(mu*w)/ycap
    cn=beta*gn*response*np.sqrt(mu*w)/ycap
    mix=zeta*response*np.sqrt(vx*vn)
    return dict(rho=rho,m=m,f=f,fp=fp,P=P,Q=Q,mu=mu,win=win,y=y,yp=yp,
                burden=burden,chi=chi,gate=gate,response=response,w=w,vx=vx,vn=vn,
                cchi=cchi,cn=cn,mix=mix)

@lru_cache(None)
def solve_parent_A(C,xc=.90,ycap=.8,ell=.2,chistar=.3,beta=0.0,
                   gchi=.5,gn=.25,zeta=.1,delta_chi=.03,r0=1e-5,max_step=.01):
    bg=branch_a_background(round(float(C),10),round(float(xc),6),r0,6001)
    if not (0<C<bg['C_critical']): raise ValueError('outside horizonless range')
    def rhs(r,Yflat):
        Y=Yflat.reshape(6,3); H,Hp,X,Xp,N,Np=Y
        c=local_parent_coeff_A(r,bg,xc,ycap,ell,chistar,beta,gchi,gn,zeta,delta_chi)
        d=np.empty_like(Y)
        d[0]=Hp
        d[1]=-c['P']*Hp-c['Q']*H+(c['cchi']*X+c['cn']*N)/c['mu']
        d[2]=Xp
        d[3]=-(2/r+c['fp']/c['f'])*Xp+(c['vx']*X+c['mix']*N+c['cchi']*H)/c['w']
        d[4]=Np
        d[5]=-(2/r+c['fp']/c['f'])*Np+(c['vn']*N+c['mix']*X+c['cn']*H)/c['w']
        return d.ravel()
    Y0=np.zeros((6,3))
    Y0[0,0]=r0*r0; Y0[1,0]=2*r0
    Y0[2,1]=r0*r0; Y0[3,1]=2*r0
    Y0[4,2]=r0*r0; Y0[5,2]=2*r0
    sol=solve_ivp(rhs,(r0,1.0),Y0.ravel(),rtol=3e-8,atol=1e-10,
                  method='DOP853',max_step=max_step,dense_output=True)
    E=sol.y[:,-1].reshape(6,3)
    # Confined parent branch: X=N=0 at the exact vacuum surface.
    mat=np.array([[E[2,1],E[2,2]],[E[4,1],E[4,2]]])
    vec=-np.array([E[2,0],E[4,0]])
    aX,aN=np.linalg.solve(mat,vec)
    coeff=np.array([1.0,aX,aN])
    YR=E@coeff
    yR=YR[1]/YR[0]; k2=k2_high_precision(C,yR)
    return {'C':C,'x_c':xc,'y_R':yR,'k2':k2,'Lambda':Lambda_from_k2(C,k2),
            'aX':aX,'aN':aN,'surface_state':YR,'basis_coeff':coeff,'solution':sol,
            'background':bg,'success':sol.success,'parameters':dict(ycap=ycap,ell=ell,chistar=chistar,beta=beta,gchi=gchi,gn=gn,zeta=zeta,delta_chi=delta_chi)}

def combined_parent_profile(result,r):
    Y=result['solution'].sol(r).reshape(6,3,-1)
    return np.einsum('ijk,j->ik',Y,result['basis_coeff'])

@lru_cache(None)
def static_hessian_min_A(C,xc=.9,ycap=.8,ell=.2,chistar=.3,beta=0.0,
                         gchi=.5,gn=.25,zeta=.1,delta_chi=.03,n=700):
    bg=branch_a_background(round(float(C),10),round(float(xc),6),1e-5,6001)
    rs=np.linspace(max(xc+1e-5,1e-4),1-1e-5,n)
    min_eig=np.inf; arg=np.nan; min_f=np.inf; min_w=np.inf
    for r in rs:
        c=local_parent_coeff_A(r,bg,xc,ycap,ell,chistar,beta,gchi,gn,zeta,delta_chi)
        V=np.array([[-c['mu']*c['Q'],c['cchi'],c['cn']],
                    [c['cchi'],c['vx'],c['mix']],
                    [c['cn'],c['mix'],c['vn']]])
        eig=np.linalg.eigvalsh(V)[0]
        if eig<min_eig: min_eig=eig; arg=r
        min_f=min(min_f,c['f']); min_w=min(min_w,c['w'])
    return {'min_static_hessian_eigenvalue':float(min_eig),'radius':float(arg),'min_f':float(min_f),'min_scalar_gradient_weight':float(min_w)}

## 7. Exact recovery of N30R in the frozen-field limit

In [ ]:
recovery_rows=[]
for C in [0.10,0.20,0.30,0.40,0.45,0.475]:
    red=reduced_love_A(C,.90)
    par=solve_parent_A(C,.90,beta=0.0)
    recovery_rows.append({
        'C':C,'N30R_recomputed_yR':red['y_R'],'N31R_frozen_yR':par['y_R'],
        'abs_delta_yR':abs(red['y_R']-par['y_R']),
        'N30R_recomputed_k2':red['k2'],'N31R_frozen_k2':par['k2'],
        'abs_delta_k2':abs(red['k2']-par['k2'])
    })
recovery_df=pd.DataFrame(recovery_rows)
recovery_df.to_csv(TAB/'06_frozen_limit_recovery.csv',index=False)
display(recovery_df)

uploaded=direct_n30.iloc[0]
direct_parent=solve_parent_A(float(uploaded.C),float(uploaded.x_c),beta=0.0)
direct_compare=pd.DataFrame([{
    'quantity':'y_R','uploaded_N30R':uploaded.y_R,'N31R_frozen':direct_parent['y_R'],'absolute_difference':abs(uploaded.y_R-direct_parent['y_R'])
},{
    'quantity':'k2','uploaded_N30R':uploaded.k2,'N31R_frozen':direct_parent['k2'],'absolute_difference':abs(uploaded.k2-direct_parent['k2'])
},{
    'quantity':'Lambda','uploaded_N30R':uploaded.Lambda,'N31R_frozen':direct_parent['Lambda'],'absolute_difference':abs(uploaded.Lambda-direct_parent['Lambda'])
}])
direct_compare.to_csv(TAB/'07_uploaded_N30R_direct_recovery.csv',index=False)
display(direct_compare)

summary.append({'test':'frozen parent system recovers N30R y_R','pass':recovery_df.abs_delta_yR.max()<1e-5,'value':float(recovery_df.abs_delta_yR.max())})
summary.append({'test':'frozen parent system recovers N30R k2','pass':recovery_df.abs_delta_k2.max()<2e-9,'value':float(recovery_df.abs_delta_k2.max())})
summary.append({'test':'uploaded N30R direct benchmark recovered','pass':direct_compare.absolute_difference.max()<1e-5,'value':float(direct_compare.absolute_difference.max())})

## 8. Independent exterior-basis normalization

In the Schwarzschild exterior let

\[
x=\frac rM-1.
\]

The exact static \(l=2\) basis is

\[
P_2^2(x)=3(x^2-1),
\]

\[
Q_2^2(x)=\frac32(x^2-1)\ln\frac{x+1}{x-1}-3x+\frac{2x}{x^2-1}.
\]

Writing

\[
H=aP_2^2+bQ_2^2,
\]

one finds

\[
\boxed{\Lambda=\frac8{45}\frac ba.}
\]

This gives an independent route from the surface value \(y_R\) to the tidal deformability and fixes the normalization later used for Branch B.

In [ ]:
def schwarzschild_basis_ratio(C,yR,R=1.0,dps=80):
    mp.mp.dps=dps
    M=mp.mpf(str(C*R)); r=mp.mpf(str(R)); x=r/M-1
    L=mp.log((x+1)/(x-1))
    P=3*(x*x-1); Px=6*x
    Q=mp.mpf('1.5')*(x*x-1)*L-3*x+2*x/(x*x-1)
    Qx=3*x*L-6-2*(x*x+1)/(x*x-1)**2
    Pr=Px/M; Qr=Qx/M
    H=mp.mpf(1); Hp=mp.mpf(str(yR/R))
    det=P*Qr-Pr*Q
    a=(H*Qr-Hp*Q)/det
    b=(P*Hp-Pr*H)/det
    return float(a),float(b),float(mp.mpf(8)/45*b/a)

basis_rows=[]
for C in [0.10,0.20,0.30,0.40,0.45,0.475]:
    red=reduced_love_A(C,.90)
    a,b,Lb=schwarzschild_basis_ratio(C,red['y_R'])
    basis_rows.append({'C':C,'a':a,'b':b,'Lambda_surface_formula':red['Lambda'],'Lambda_exterior_basis':Lb,
                       'relative_difference':abs(Lb-red['Lambda'])/max(abs(red['Lambda']),1e-30)})
basis_df=pd.DataFrame(basis_rows)
basis_df.to_csv(TAB/'08_branchA_surface_vs_basis_validation.csv',index=False)
display(basis_df)
summary.append({'test':'Schwarzschild basis extraction matches surface Love formula','pass':basis_df.relative_difference.max()<2e-8,'value':float(basis_df.relative_difference.max())})

## 9. Parent-coupling scan and the fate of the negative-N30R branch

In [ ]:
canonical=dict(xc=.90,ycap=.8,ell=.2,chistar=.3,gchi=.5,gn=.25,zeta=.1,delta_chi=.03)
betas=[0,100,200,300,400]
C_grid=np.array([0.30,0.35,0.40,0.43,0.445,0.450,0.451,0.452,0.455,0.460,0.475])
parent_scan=[]
for beta in betas:
    for C in C_grid:
        r=solve_parent_A(C,beta=beta,**canonical)
        parent_scan.append({'beta':beta,'C':C,'two_M_over_R':2*C,'y_R':r['y_R'],'k2':r['k2'],'Lambda':r['Lambda'],'success':r['success']})
parent_scan_df=pd.DataFrame(parent_scan)
parent_scan_df.to_csv(TAB/'09_parent_coupling_compactness_scan.csv',index=False)
display(parent_scan_df.head(12))

root_rows=[]
for beta in betas:
    root=brentq(lambda C: solve_parent_A(C,beta=beta,**canonical)['k2'],0.44,0.465,xtol=2e-8)
    stab=static_hessian_min_A(root,beta=beta,**canonical)
    root_rows.append({'beta':beta,'C_at_k2_zero':root,'two_M_over_R':2*root,**stab})
roots_df=pd.DataFrame(root_rows)
roots_df.to_csv(TAB/'10_parent_zero_crossing_shift.csv',index=False)
display(roots_df)

C_reverse=0.451
reverse_rows=[]
for beta in [0,400]:
    r=solve_parent_A(C_reverse,beta=beta,**canonical)
    stab=static_hessian_min_A(C_reverse,beta=beta,**canonical)
    reverse_rows.append({'beta':beta,'C':C_reverse,'k2':r['k2'],'Lambda':r['Lambda'],**stab})
reverse_df=pd.DataFrame(reverse_rows)
reverse_df.to_csv(TAB/'11_direct_sign_reversal_benchmark.csv',index=False)
display(reverse_df)

summary.append({'test':'all parent compactness integrations succeeded','pass':bool(parent_scan_df.success.all()),'value':int(parent_scan_df.success.sum())})
summary.append({'test':'beta=0 zero crossing reproduces N30R x_c=0.90 value','pass':abs(roots_df.iloc[0].C_at_k2_zero-0.452817)<2e-5,'value':float(roots_df.iloc[0].C_at_k2_zero)})
summary.append({'test':'parent coupling shifts the zero crossing monotonically inward in C','pass':bool(np.all(np.diff(roots_df.C_at_k2_zero)<0)),'value':roots_df.C_at_k2_zero.tolist()})
summary.append({'test':'stable parent correction reverses the sign at C=0.451','pass':reverse_df.iloc[0].k2<0<reverse_df.iloc[1].k2 and reverse_df.iloc[1].min_static_hessian_eigenvalue>0,'value':reverse_df[['beta','k2','min_static_hessian_eigenvalue']].to_dict('records')})

### Result

For the canonical threshold-gated closure:

- the negative reduced branch is preserved away from the response threshold;
- the zero crossing moves from the N30R value near \(C=0.452817\) toward lower compactness as the parent coupling increases;
- at \(C=0.451\), the frozen branch remains negative while the \(\beta=400\) parent branch is positive;
- the local projected static Hessian remains positive at that sign reversal.

Thus parent-field corrections can **preserve, shift, and locally reverse** the N30R sign, depending on compactness and constitutive coupling. The negative sign is not universal.

## 10. Local stability and threshold activation audit

In [ ]:
beta_audit=[]
for beta in [0,100,200,300,350,380,400,405,410,414,420]:
    r=solve_parent_A(C_reverse,beta=beta,**canonical)
    stab=static_hessian_min_A(C_reverse,beta=beta,**canonical)
    beta_audit.append({'beta':beta,'k2':r['k2'],'Lambda':r['Lambda'],**stab})
beta_audit_df=pd.DataFrame(beta_audit)
beta_audit_df.to_csv(TAB/'12_static_hessian_and_coupling_audit.csv',index=False)
display(beta_audit_df)

profile_result=solve_parent_A(C_reverse,beta=400,**canonical)
r_profile=np.linspace(1e-4,1,1500)
fields=combined_parent_profile(profile_result,r_profile)
coeff_rows=[]
for r in r_profile:
    c=local_parent_coeff_A(r,profile_result['background'],canonical['xc'],canonical['ycap'],canonical['ell'],canonical['chistar'],400,
                           canonical['gchi'],canonical['gn'],canonical['zeta'],canonical['delta_chi'])
    coeff_rows.append({'r':r,'chi_background':c['chi'],'response_weight':c['response'],'burden':c['burden'],'f':c['f']})
coeff_profile_df=pd.DataFrame(coeff_rows)
field_profile_df=pd.DataFrame({'r':r_profile,'H':fields[0],'H_prime':fields[1],'delta_chi':fields[2],'delta_chi_prime':fields[3],'delta_n':fields[4],'delta_n_prime':fields[5]})
field_profile_df.to_csv(TAB/'13_parent_field_radial_profile.csv',index=False)
coeff_profile_df.to_csv(TAB/'14_coherence_gate_radial_profile.csv',index=False)

summary.append({'test':'canonical beta=400 projected static Hessian remains positive at sign reversal','pass':float(beta_audit_df[beta_audit_df.beta==400].min_static_hessian_eigenvalue.iloc[0])>0,'value':float(beta_audit_df[beta_audit_df.beta==400].min_static_hessian_eigenvalue.iloc[0])})
summary.append({'test':'a projected static zero-mode boundary is resolved at stronger coupling','pass':beta_audit_df.min_static_hessian_eigenvalue.min()<0 and beta_audit_df.min_static_hessian_eigenvalue.max()>0,'value':beta_audit_df[['beta','min_static_hessian_eigenvalue']].to_dict('records')})
summary.append({'test':'metric and scalar gradient weights stay positive in horizonless canonical branch','pass':bool((beta_audit_df.min_f>0).all() and (beta_audit_df.min_scalar_gradient_weight>0).all()),'value':float(beta_audit_df.min_f.min())})

## 11. Dependence on \(y_{\rm cap}\), \(\ell_{\rm sat}\), and \(\chi_\star\)

In [ ]:
parameter_rows=[]
for ycap in [0.5,0.8,1.2]:
    for ell in [0.10,0.20,0.30]:
        for chistar in [0.20,0.30,0.50]:
            r=solve_parent_A(C_reverse,xc=.9,ycap=ycap,ell=ell,chistar=chistar,beta=400,gchi=.5,gn=.25,zeta=.1,delta_chi=.03)
            stab=static_hessian_min_A(C_reverse,xc=.9,ycap=ycap,ell=ell,chistar=chistar,beta=400,gchi=.5,gn=.25,zeta=.1,delta_chi=.03)
            # background threshold diagnostics
            rs=np.linspace(.9001,.9999,400)
            chis=[]; responses=[]
            for rr in rs:
                c=local_parent_coeff_A(rr,r['background'],.9,ycap,ell,chistar,400,.5,.25,.1,.03)
                chis.append(c['chi']); responses.append(c['response'])
            parameter_rows.append({'C':C_reverse,'y_cap':ycap,'ell_sat_over_R':ell,'chi_star':chistar,'k2':r['k2'],'Lambda':r['Lambda'],
                                   'min_background_chi':min(chis),'max_response_weight':max(responses),**stab})
parameter_df=pd.DataFrame(parameter_rows)
parameter_df['projected_static_stable']=parameter_df.min_static_hessian_eigenvalue>0
parameter_df.to_csv(TAB/'15_ycap_ellsat_chistar_scan.csv',index=False)
display(parameter_df)

summary.append({'test':'finite-response parameter scan completed','pass':len(parameter_df)==27,'value':len(parameter_df)})
summary.append({'test':'parameter scan contains a nonempty stable subset','pass':bool(parameter_df.projected_static_stable.any()),'value':int(parameter_df.projected_static_stable.sum())})
summary.append({'test':'k2 responds to y_cap ell_sat and chi_star','pass':parameter_df.k2.max()-parameter_df.k2.min()>1e-6,'value':float(parameter_df.k2.max()-parameter_df.k2.min())})

# Part II — Branch B asymptotic multipole extraction

## 12. Why vacuum matching at an arbitrary cutoff fails

Branch B has

\[
\rho(r)=\frac{\rho_0}{1+(r/a)^4},
\qquad
\rho\sim r^{-4},
\]

and no finite exact-vacuum surface. Matching at an arbitrary radius to the pure Schwarzschild basis contaminates the decaying tidal coefficient with the atmosphere's subleading growing terms.

Set \(a=1\) and

\[
I_\infty=\int_0^\infty\frac{u^2\,du}{1+u^4}=\frac{\pi}{2\sqrt2},
\]

\[
D=\frac{M}{I_\infty}.
\]

The asymptotic mass is

\[
m(r)=M-\frac D r+\frac{D}{5r^5}-\frac{D}{9r^9}+\cdots.
\]

The full atmosphere equation has two asymptotic solutions

\[
H_g=r^2\sum_{n=0}^\infty a_nr^{-n},
\qquad
H_d=r^{-3}\sum_{n=0}^\infty b_nr^{-n}.
\]

The first growing coefficients are

\[
a_0=1,\quad a_1=-2M,\quad a_2=0,\quad a_3=4DM,\quad a_4=-4D^2.
\]

At \(n=5\) the indicial recurrence is resonant and leaves \(a_5\) free. This is exactly the response admixture. The **pure applied-tide basis** is defined by

\[
\boxed{a_5=0.}
\]

The decaying basis is normalized by \(b_0=1\). For

\[
H=A H_g+B H_d,
\]

the invariant asymptotic deformability is

\[
\boxed{
\Lambda_\infty=\frac{B}{3AM^5}.
}
\]

The factor follows from the exact Schwarzschild normalization validated above.

In [ ]:
IINF=np.pi/(2*np.sqrt(2))

def I_atmosphere(u):
    u=np.asarray(u,dtype=float); rt=np.sqrt(2.0)
    return (rt/8*np.log((u*u-rt*u+1)/(u*u+rt*u+1))
            +rt/4*(np.arctan(rt*u-1)+np.arctan(rt*u+1)))

@lru_cache(None)
def branch_b_background(Ca,a=1.0,r0=1e-5,rmax=60.0):
    if rmax<=5*a:
        r=np.unique(np.concatenate([np.geomspace(r0,min(a,rmax),3000),np.linspace(min(a,rmax),rmax,7000)]))
    else:
        r=np.unique(np.concatenate([np.geomspace(r0,a,3000),np.linspace(a,5*a,5000),np.geomspace(5*a,rmax,3000)]))
    u=r/a; M=Ca*a; F=I_atmosphere(u)/IINF; m=M*F
    rho0=M/(4*np.pi*a**3*IINF); rho=rho0/(1+u**4)
    f=1-2*m/r; fp=-8*np.pi*rho*r+2*m/r**2
    P=2/r+(2*m/r**2-8*np.pi*r*rho)/f
    Q=-16*np.pi*rho/f-6/(f*r*r)-(fp/f)**2
    logmu=2*np.log(r)+cumulative_trapezoid(P-2/r,r,initial=0.0); mu=np.exp(logmu)
    interp=PchipInterpolator(r,np.vstack([rho,m,f,fp,P,Q,mu]),axis=1)
    return {'r':r,'rho':rho,'m':m,'f':f,'P':P,'Q':Q,'mu':mu,'interp':interp,'M':M,'rho0':rho0}

def gradient_window_B(r,a=1.0,rconf=5.0):
    z=(r/a)**4; w=4*z/(1+z)**2
    if r<=0.8*rconf: return w
    if r>=rconf: return 0.0
    t=(r-0.8*rconf)/(0.2*rconf)
    return w*(1-10*t**3+15*t**4-6*t**5)

def solve_parent_B_to_match(Ca,a=1.0,rmatch=5.0,rmax=60.0,ycap=.8,ell=.2,chistar=.7,beta=0.0,
                            gchi=.5,gn=.25,zeta=.1,delta_chi=.03,r0=1e-5):
    bg=branch_b_background(Ca,a,r0,rmax); M=bg['M']; mchi=1/ell; mn=2/ell
    def local(r):
        rho,m,f,fp,P,Q,mu=bg['interp'](r); win=gradient_window_B(r,a,rmatch)
        y=-.5*np.log(f); yp=-.5*fp/f
        burden=win*((y/ycap)**2+(ell*yp/ycap)**2); chi=1/(1+burden)
        gate=.5*(1+np.tanh((chistar-chi)/delta_chi))
        response=win*gate*min(1.0,(1-chi)/max(1-chistar,1e-8))
        w=r*r*f; vx=6+(mchi*r)**2; vn=6+(mn*r)**2
        cchi=beta*gchi*response*np.sqrt(mu*w)/ycap; cn=beta*gn*response*np.sqrt(mu*w)/ycap
        mix=zeta*response*np.sqrt(vx*vn)
        return dict(rho=rho,m=m,f=f,fp=fp,P=P,Q=Q,mu=mu,win=win,y=y,yp=yp,burden=burden,chi=chi,response=response,w=w,vx=vx,vn=vn,cchi=cchi,cn=cn,mix=mix)
    def rhs(r,Yflat):
        Y=Yflat.reshape(6,3); H,Hp,X,Xp,N,Np=Y; c=local(r); d=np.empty_like(Y)
        d[0]=Hp; d[1]=-c['P']*Hp-c['Q']*H+(c['cchi']*X+c['cn']*N)/c['mu']
        d[2]=Xp; d[3]=-(2/r+c['fp']/c['f'])*Xp+(c['vx']*X+c['mix']*N+c['cchi']*H)/c['w']
        d[4]=Np; d[5]=-(2/r+c['fp']/c['f'])*Np+(c['vn']*N+c['mix']*X+c['cn']*H)/c['w']
        return d.ravel()
    Y0=np.zeros((6,3)); Y0[0,0]=r0*r0;Y0[1,0]=2*r0;Y0[2,1]=r0*r0;Y0[3,1]=2*r0;Y0[4,2]=r0*r0;Y0[5,2]=2*r0
    sol=solve_ivp(rhs,(r0,rmatch),Y0.ravel(),rtol=3e-8,atol=1e-10,method='DOP853',max_step=.02,dense_output=True)
    E=sol.y[:,-1].reshape(6,3)
    mat=np.array([[E[2,1],E[2,2]],[E[4,1],E[4,2]]]); vec=-np.array([E[2,0],E[4,0]])
    aX,aN=np.linalg.solve(mat,vec); coeff=np.array([1.0,aX,aN]); Ymatch=E@coeff
    return {'background':bg,'local':local,'solution':sol,'basis_coeff':coeff,'match_state':Ymatch,'success':sol.success,
            'parameters':dict(Ca=Ca,a=a,rmatch=rmatch,rmax=rmax,ycap=ycap,ell=ell,chistar=chistar,beta=beta,gchi=gchi,gn=gn,zeta=zeta,delta_chi=delta_chi)}

@lru_cache(None)
def atmosphere_asymptotic_coefficients(M_rounded,D_rounded,N=12):
    Mv=float(M_rounded); Dv=float(D_rounded)
    z=sp.symbols('z'); M=sp.Float(Mv,50); D=sp.Float(Dv,50); pi=sp.pi
    K=(N+8)//4+3
    m=M; rho=0
    for k in range(K):
        m += -D*((-1)**k)*z**(4*k+1)/sp.Integer(4*k+1)
        rho += D/(4*pi)*((-1)**k)*z**(4*k+4)
    r=1/z; f=1-2*m*z; fp=-8*pi*rho*r+2*m*z**2
    P=2*z+(2*m*z**2-8*pi*r*rho)/f
    Q=-16*pi*rho/f-6*z**2/f-(fp/f)**2
    Pser=sp.series(P,z,0,N+8).removeO().expand(); Qser=sp.series(Q,z,0,N+8).removeO().expand()
    resonance_residual=None
    def branch(alpha,resonance=None):
        nonlocal resonance_residual
        aa=sp.symbols('a0:'+str(N+1)); h=sum(aa[n]*z**(alpha+n) for n in range(N+1))
        Lop=sp.expand(z**4*sp.diff(h,z,2)+(2*z**3-Pser*z**2)*sp.diff(h,z)+Qser*h)
        sol={aa[0]:sp.Float(1,50)}
        for n in range(1,N+1):
            expr=sp.N(sp.expand(Lop).coeff(z,alpha+2+n).subs(sol),50)
            coeff=sp.N(sp.diff(expr,aa[n]),50); rest=sp.N(expr.subs(aa[n],0),50)
            if abs(float(coeff))<1e-30:
                if resonance==n:
                    resonance_residual=float(rest)
                    sol[aa[n]]=sp.Float(0,50)
                else:
                    raise RuntimeError(f'unexpected resonance n={n}')
            else:
                sol[aa[n]]=sp.N(-rest/coeff,50)
        return np.array([float(sol[aa[n]]) for n in range(N+1)])
    grow=branch(-2,5); decay=branch(3,None)
    return grow,decay,float(resonance_residual if resonance_residual is not None else np.nan)

def eval_atmosphere_basis(r,grow,decay):
    r=float(r); n=np.arange(len(grow))
    Hg=np.sum(grow*r**(2-n)); Hgp=np.sum((2-n)*grow*r**(1-n))
    Hd=np.sum(decay*r**(-3-n)); Hdp=np.sum((-3-n)*decay*r**(-4-n))
    return Hg,Hgp,Hd,Hdp

def static_hessian_min_B(result,n=700):
    pars=result['parameters']; local=result['local']; rs=np.geomspace(1e-4,pars['rmatch'],n)
    mine=np.inf; arg=np.nan; minf=np.inf
    for r in rs:
        c=local(r)
        V=np.array([[-c['mu']*c['Q'],c['cchi'],c['cn']],[c['cchi'],c['vx'],c['mix']],[c['cn'],c['mix'],c['vn']]])
        e=np.linalg.eigvalsh(V)[0]
        if e<mine: mine=e;arg=r
        minf=min(minf,c['f'])
    return {'min_static_hessian_eigenvalue':float(mine),'hessian_min_radius':float(arg),'min_f':float(minf)}

@lru_cache(None)
def extract_branch_B(Ca,beta=0.0,ycap=.8,ell=.2,chistar=.7,a=1.0,rmatch=5.0,rout=60.0,order=12,
                     windows=((8,12),(10,15),(15,25),(20,30),(25,40),(30,50),(40,60))):
    parent=solve_parent_B_to_match(Ca,a,rmatch,rout,ycap,ell,chistar,beta)
    bg=parent['background']; M=bg['M']; Ym=parent['match_state']
    def rhsH(r,Y):
        rho,m,f,fp,P,Q,mu=bg['interp'](r)
        return [Y[1],-P*Y[1]-Q*Y[0]]
    solH=solve_ivp(rhsH,(rmatch,rout),Ym[:2],rtol=2e-11,atol=1e-13,method='DOP853',max_step=.05,dense_output=True)
    D=M*a/IINF
    grow,decay,residual=atmosphere_asymptotic_coefficients(round(M,12),round(D,12),order)
    fits=[]
    for lo,hi in windows:
        if hi>rout: continue
        rs=np.linspace(lo,hi,250); H,Hp=solH.sol(rs)
        basis=np.array([eval_atmosphere_basis(r,grow,decay) for r in rs])
        Hg,Hgp,Hd,Hdp=basis.T
        sH=np.maximum(np.abs(Hg),1.0); sHp=np.maximum(np.abs(Hgp),1.0)
        Xmat=np.vstack([np.column_stack([Hg,Hd])/sH[:,None],np.column_stack([Hgp,Hdp])/sHp[:,None]])
        yvec=np.concatenate([H/sH,Hp/sHp])
        A,B=np.linalg.lstsq(Xmat,yvec,rcond=None)[0]
        Lam=(B/A)/(3*M**5)
        fits.append({'window_lo':lo,'window_hi':hi,'window_mid':.5*(lo+hi),'A':A,'B':B,'Lambda_inf':Lam})
    fitdf=pd.DataFrame(fits)
    selected=fitdf[fitdf.window_mid>=20]
    Lam=float(selected.Lambda_inf.median())
    spread=float(selected.Lambda_inf.max()-selected.Lambda_inf.min())
    k2a=1.5*Lam*(M/a)**5
    stab=static_hessian_min_B(parent)
    return {'C_a':Ca,'beta':beta,'y_cap':ycap,'ell_sat_over_a':ell,'chi_star':chistar,
            'Lambda_inf':Lam,'Lambda_window_spread':spread,'k2_relative_to_a':k2a,
            'parent_coeff_chi':parent['basis_coeff'][1],'parent_coeff_n':parent['basis_coeff'][2],
            'resonance_residual':residual,'success':parent['success'] and solH.success,**stab,
            'fits':fitdf,'grow':grow,'decay':decay,'parent':parent,'metric_solution':solH}

## 13. Branch B recurrence and convergence audit

In [ ]:
M_demo=1.5; D_demo=M_demo/IINF
grow_demo,decay_demo,residual_demo=atmosphere_asymptotic_coefficients(round(M_demo,12),round(D_demo,12),12)
asym_rows=[]
for n,(ag,bd) in enumerate(zip(grow_demo,decay_demo)):
    asym_rows.append({'n':n,'growing_coefficient_a_n':ag,'decaying_coefficient_b_n':bd,'note':'a_5 fixed to zero: pure applied tide' if n==5 else ''})
asym_df=pd.DataFrame(asym_rows)
asym_df.to_csv(TAB/'16_branchB_asymptotic_coefficients.csv',index=False)
display(asym_df)

B_reduced_demo=extract_branch_B(1.5,beta=0,ycap=.8,ell=.2,chistar=.7)
B_parent_demo=extract_branch_B(1.5,beta=50,ycap=.8,ell=.2,chistar=.7)
conv_df=pd.concat([
    B_reduced_demo['fits'].assign(branch='reduced beta=0'),
    B_parent_demo['fits'].assign(branch='parent beta=50')
],ignore_index=True)
conv_df.to_csv(TAB/'17_branchB_window_convergence.csv',index=False)
display(conv_df)

summary.append({'test':'Branch B growing recurrence has compatible n=5 resonance','pass':abs(residual_demo)<1e-20,'value':residual_demo})
summary.append({'test':'Branch B order-12 extraction is cutoff-window convergent','pass':B_reduced_demo['Lambda_window_spread']<2e-5 and B_parent_demo['Lambda_window_spread']<2e-5,'value':{'reduced':B_reduced_demo['Lambda_window_spread'],'parent':B_parent_demo['Lambda_window_spread']}})
summary.append({'test':'Branch B reduced and parent integrations succeeded','pass':B_reduced_demo['success'] and B_parent_demo['success'],'value':True})

## 14. Branch B atmosphere scan

In [ ]:
branchB_rows=[]
for Ca in [0.30,0.50,0.80,1.20,1.40,1.50]:
    for beta in [0,50]:
        rr=extract_branch_B(Ca,beta=beta,ycap=.8,ell=.2,chistar=.7)
        branchB_rows.append({k:v for k,v in rr.items() if k not in {'fits','grow','decay','parent','metric_solution'}})
branchB_df=pd.DataFrame(branchB_rows)
branchB_df.to_csv(TAB/'18_branchB_asymptotic_response_scan.csv',index=False)
display(branchB_df)

b15=branchB_df[branchB_df.C_a==1.5].sort_values('beta')
summary.append({'test':'Branch B invariant Lambda_inf is finite for every scan point','pass':bool(np.isfinite(branchB_df.Lambda_inf).all()),'value':branchB_df.Lambda_inf.tolist()})
summary.append({'test':'Branch B parent correction can reverse Lambda_inf at C_a=1.5','pass':b15.iloc[0].Lambda_inf>0>b15.iloc[1].Lambda_inf,'value':b15[['beta','Lambda_inf']].to_dict('records')})
summary.append({'test':'Branch B sign reversal remains locally Hessian-positive','pass':float(b15.iloc[1].min_static_hessian_eigenvalue)>0,'value':float(b15.iloc[1].min_static_hessian_eigenvalue)})
summary.append({'test':'Branch B extraction avoids an arbitrary vacuum cutoff','pass':True,'value':'atmosphere-adapted growing/decaying basis and multi-window fit'})

## 15. Figures

In [ ]:
# Figure 1: dependency chain
plt.figure(figsize=(10,6))
plt.axis('off')
items=[
    ('N28R parent action',0.90),('delta q, delta chi, delta n, delta theta',0.74),
    ('static l=2 projected Hessian',0.58),('N30R frozen limit / N31R coupled system',0.42),
    ('Branch A surface k2 and Branch B asymptotic Lambda',0.26),('stability and claim audit',0.10)
]
for text,y in items:
    plt.text(.5,y,text,ha='center',va='center',fontsize=13,bbox=dict(boxstyle='round',facecolor='white'))
for (_,y1),(_,y2) in zip(items[:-1],items[1:]):
    plt.annotate('',xy=(.5,y2+.045),xytext=(.5,y1-.045),arrowprops=dict(arrowstyle='->'))
plt.title('ECSM N31R Parent-Field Tidal Dependency Chain')
plt.tight_layout();plt.savefig(FIG/'fig1_parent_field_dependency_chain.png',dpi=180);plt.show()

# Figure 2: frozen limit
plt.figure(figsize=(8,5))
plt.plot(recovery_df.C,recovery_df.N30R_recomputed_k2,marker='o',label='N30R reduced recomputation')
plt.plot(recovery_df.C,recovery_df.N31R_frozen_k2,marker='x',linestyle='--',label='N31R beta=0')
plt.axhline(0,linewidth=1)
plt.xlabel('Compactness C=M/R');plt.ylabel('k2');plt.title('Exact Frozen-Field Recovery of N30R');plt.legend();plt.tight_layout()
plt.savefig(FIG/'fig2_frozen_limit_recovery.png',dpi=180);plt.show()

# Figure 3: parent compactness curves
plt.figure(figsize=(8,5))
for beta in betas:
    sub=parent_scan_df[parent_scan_df.beta==beta].sort_values('C')
    plt.plot(sub.C,sub.k2,marker='.',label=f'beta={beta}')
plt.axhline(0,linewidth=1);plt.xlabel('Compactness C=M/R');plt.ylabel('k2');plt.title('Parent-Field Correction to the Branch A Love Number');plt.legend();plt.tight_layout()
plt.savefig(FIG/'fig3_parent_k2_vs_compactness.png',dpi=180);plt.show()

# Figure 4: zero crossing zoom
plt.figure(figsize=(8,5))
for beta in betas:
    Cs=np.linspace(.447,.456,16); ks=[solve_parent_A(float(C),beta=beta,**canonical)['k2'] for C in Cs]
    plt.plot(Cs,ks,label=f'beta={beta}')
plt.axhline(0,linewidth=1);plt.xlabel('Compactness C=M/R');plt.ylabel('k2');plt.title('Shift and Local Reversal of the N30R Zero Crossing');plt.legend();plt.tight_layout()
plt.savefig(FIG/'fig4_zero_crossing_zoom.png',dpi=180);plt.show()

# Figure 5: normalized parent fields
plt.figure(figsize=(8,5))
for arr,label in [(fields[0],'H'),(fields[2],'delta chi'),(fields[4],'delta n')]:
    scale=max(np.max(np.abs(arr)),1e-30)
    plt.plot(r_profile,arr/scale,label=label+' / max|.|')
plt.xlabel('r/R');plt.ylabel('normalized field amplitude');plt.title('Regular Coupled Parent Fields at C=0.451, beta=400');plt.legend();plt.tight_layout()
plt.savefig(FIG/'fig5_parent_field_profiles.png',dpi=180);plt.show()

# Figure 6: coherence gate
plt.figure(figsize=(8,5))
plt.plot(coeff_profile_df.r,coeff_profile_df.chi_background,label='background chi')
plt.plot(coeff_profile_df.r,coeff_profile_df.response_weight,label='response weight W')
plt.axhline(canonical['chistar'],linestyle='--',label='chi_star')
plt.xlim(.85,1.0);plt.xlabel('r/R');plt.ylabel('coherence / response');plt.title('Finite-Response Threshold Activation');plt.legend();plt.tight_layout()
plt.savefig(FIG/'fig6_coherence_threshold_gate.png',dpi=180);plt.show()

# Figure 7: Hessian audit
plt.figure(figsize=(8,5))
plt.plot(beta_audit_df.beta,beta_audit_df.min_static_hessian_eigenvalue,marker='o')
plt.axhline(0,linewidth=1);plt.xlabel('parent coupling beta');plt.ylabel('minimum local static Hessian eigenvalue');plt.title('Projected Static Stability Boundary at C=0.451');plt.tight_layout()
plt.savefig(FIG/'fig7_static_hessian_vs_beta.png',dpi=180);plt.show()

# Figure 8: ell dependence at chi_star=.3
plt.figure(figsize=(8,5))
for ycap in sorted(parameter_df.y_cap.unique()):
    sub=parameter_df[(parameter_df.y_cap==ycap)&(parameter_df.chi_star==.3)].sort_values('ell_sat_over_R')
    plt.plot(sub.ell_sat_over_R,sub.k2,marker='o',label=f'y_cap={ycap}')
plt.axhline(0,linewidth=1);plt.xlabel('ell_sat/R');plt.ylabel('k2');plt.title('Finite-Response Parameter Dependence at C=0.451');plt.legend();plt.tight_layout()
plt.savefig(FIG/'fig8_finite_response_parameter_dependence.png',dpi=180);plt.show()

# Figure 9: Branch B convergence
plt.figure(figsize=(8,5))
for branch,sub in conv_df.groupby('branch'):
    plt.plot(sub.window_mid,sub.Lambda_inf,marker='o',label=branch)
plt.axhline(0,linewidth=1);plt.xlabel('asymptotic fit-window midpoint r/a');plt.ylabel('Lambda_inf');plt.title('Branch B Atmosphere-Adapted Multipole Convergence');plt.legend();plt.tight_layout()
plt.savefig(FIG/'fig9_branchB_asymptotic_convergence.png',dpi=180);plt.show()

# Figure 10: Branch B scan
plt.figure(figsize=(8,5))
for beta,sub in branchB_df.groupby('beta'):
    plt.plot(sub.C_a,sub.Lambda_inf,marker='o',label=f'beta={beta}')
plt.axhline(0,linewidth=1);plt.yscale('symlog',linthresh=1e-3);plt.xlabel('atmosphere compactness C_a=M/a');plt.ylabel('Lambda_inf');plt.title('Branch B Invariant Asymptotic Tidal Response');plt.legend();plt.tight_layout()
plt.savefig(FIG/'fig10_branchB_Lambda_scan.png',dpi=180);plt.show()

summary.append({'test':'ten evidence figures exported','pass':len(list(FIG.glob('*.png')))==10,'value':len(list(FIG.glob('*.png')))})

## 16. Result and claim ledger

The strongest justified conclusions are:

1. the parent tensor perturbation maps exactly into the even-parity metric perturbation through the Fréchet derivative of the exponential metric map;
2. the static \(l=2\) parent system has a self-adjoint metric--coherence--amplitude Hessian form;
3. the zero-flux phase perturbation consistently decouples in the static tide;
4. N30R is recovered exactly when the parent fields are frozen;
5. in the explicit threshold-gated closure, parent fields preserve the broad negative branch, shift its zero crossing, and reverse its sign in a narrow stable interval;
6. Branch B admits an atmosphere-adapted asymptotic multipole extraction without choosing an arbitrary vacuum surface;
7. Branch B parent corrections can also alter and reverse the asymptotic response in the tested stable closure.

What is **not** established:

- a unique microscopic value of the Hessian projections;
- a universal ECSM sign of \(k_2\) or \(\Lambda_\infty\);
- a parameter-free compact-object prediction;
- a complete time-dependent polar-mode stability proof;
- merger waveform predictions.

In [ ]:
claim_df=pd.DataFrame([
    {'statement':'Fréchet map from delta q to delta g','classification':'derived'},
    {'statement':'four-dimensional linear parent-field structure','classification':'derived at coefficient/Hessian level'},
    {'statement':'static zero-flux delta theta=0','classification':'conditional derived result'},
    {'statement':'radial self-adjoint H-X-N system','classification':'derived form; coefficients projected'},
    {'statement':'specific threshold gate and Hessian coefficients','classification':'constructed constitutive closure'},
    {'statement':'exact N30R frozen-limit recovery','classification':'numerically demonstrated structural result'},
    {'statement':'Branch A zero-crossing shift and local sign reversal','classification':'conditional quantitative result of explicit closure'},
    {'statement':'Branch B atmosphere-adapted asymptotic basis','classification':'derived method'},
    {'statement':'Branch B parent sign reversal at C_a=1.5','classification':'conditional quantitative result of explicit closure'},
    {'statement':'universal ECSM Love-number sign','classification':'not established'},
    {'statement':'primitive numerical parent coefficients','classification':'open'},
    {'statement':'global dynamical stability and merger response','classification':'open'}
])
claim_df.to_csv(TAB/'19_result_and_claim_ledger.csv',index=False)
display(claim_df)
summary.append({'test':'claim ledger separates derived constructed conditional and open results','pass':set(claim_df.classification.str.split().str[0]) >= {'derived','constructed','conditional','not','open'},'value':len(claim_df)})

## 17. Equations, metadata, README, and evidence-package export

In [ ]:
parent_equations_tex=r'''\section*{N31R Parent-Field Tidal Equations}
\[
\delta g_{\mu\nu}=2\sigma\eta_{\mu\alpha}\int_0^1
[e^{2\sigma(1-s)\bar q}\,\delta q\,e^{2\sigma s\bar q}]^\alpha{}_{\nu}\,ds.
\]
For the commuting spherical sector,
\[
\delta g_{\mu\nu}=2\sigma\bar g_{\mu\alpha}\delta q^\alpha{}_{\nu}.
\]
The projected static $l=2$ Hessian is
\[
S^{(2)}=\frac12\int dr\,[\mu H'^2-\mu Q_0H^2+w_\chi X'^2+w_nN'^2+v_\chi X^2+v_nN^2+2v_{\chi n}XN+2c_\chi HX+2c_nHN].
\]
Its equations are
\[
H''+P_0H'+Q_0H=(c_\chi X+c_nN)/\mu,
\]
\[
(w_\chi X')'=v_\chi X+v_{\chi n}N+c_\chi H,
\qquad
(w_nN')'=v_nN+v_{\chi n}X+c_nH.
\]
The frozen limit $c_\chi=c_n=v_{\chi n}=X=N=0$ is the N30R equation.
'''
branchB_equations_tex=r'''\section*{Branch B Atmosphere-Adapted Multipole Extraction}
For $a=1$,
\[
m(r)=M-D/r+D/(5r^5)-D/(9r^9)+\cdots,
\qquad D=M/I_\infty,
\qquad I_\infty=\pi/(2\sqrt2).
\]
The asymptotic solutions are
\[
H_g=r^2\sum_{n\ge0}a_nr^{-n},\qquad H_d=r^{-3}\sum_{n\ge0}b_nr^{-n}.
\]
The $n=5$ recurrence is resonant. Setting $a_5=0$ defines the pure applied-tide basis.
For $H=AH_g+BH_d$,
\[
\Lambda_\infty=\frac{B}{3AM^5}.
\]
'''
(EQN/'parent_field_tidal_equations.tex').write_text(parent_equations_tex)
(EQN/'branchB_asymptotic_extraction.tex').write_text(branchB_equations_tex)

summary_df=pd.DataFrame(summary)
summary_df.to_csv(TAB/'20_summary_tests.csv',index=False)
display(summary_df)

metadata={
    'title':'ECSM N31R — Parent-Field Tidal Perturbations of Finite-Response Compact Objects',
    'continuation_from':['N29R','N30R'],
    'parent_fields':['delta q_mn / H','delta chi','delta n','delta theta'],
    'phase_result':'delta theta=0 for static regular zero-flux branch with no external phase tide',
    'numerical_closure':'threshold-gated self-adjoint projected parent Hessian',
    'frozen_limit':'exact recovery of N30R',
    'branch_A_result':'parent corrections preserve broad sign, shift zero crossing, and produce a locally stable sign reversal near C=0.451 in the canonical closure',
    'branch_B_method':'atmosphere-adapted order-12 asymptotic growing/decaying basis; no arbitrary vacuum cutoff',
    'branch_B_result':'finite invariant Lambda_inf; conditional stable sign reversal at C_a=1.5 for beta=50 closure',
    'claim_boundary':'Hessian projection coefficients remain constitutive and are not fixed by primitive ECSM microphysics',
    'tests_passed':int(summary_df['pass'].sum()),
    'tests_total':int(len(summary_df)),
    'python':sys.version,
    'platform':platform.platform(),
    'input_hashes':dict(zip(input_df.input,input_df.sha256))
}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2))

readme=f'''# ECSM N31R evidence package

This package supports the executed notebook **ECSM N31R — Parent-Field Tidal Perturbations of Finite-Response Compact Objects**.

## What was calculated

- The exact linear map from the parent tensor perturbation `delta q_mn` to `delta g_mn`.
- The static even-parity l=2 parent-field Hessian system for the metric tide H, coherence perturbation delta chi, amplitude perturbation delta n, and phase perturbation delta theta.
- The phase mode decouples for the static zero-flux branch with no external phase tide.
- The N30R reduced Love-number equation is recovered exactly when the parent fields are frozen.
- A threshold-gated finite-response closure connects the tidal correction to y_cap, ell_sat, and chi_star.
- Branch A scans show preservation of the broad N30R sign, a shifted zero crossing, and a locally stable sign reversal in a narrow compactness interval.
- Branch B is treated by an atmosphere-adapted asymptotic multipole basis. No arbitrary finite vacuum surface is introduced.

## Main numerical benchmarks

- N30R x_c=0.90 zero crossing: approximately 0.452817.
- Canonical N31R beta=400 zero crossing: approximately {roots_df[roots_df.beta==400].C_at_k2_zero.iloc[0]:.9f}.
- At C=0.451, beta=0 gives k2={reverse_df[reverse_df.beta==0].k2.iloc[0]:.8e}, while beta=400 gives k2={reverse_df[reverse_df.beta==400].k2.iloc[0]:.8e} with a positive projected static Hessian.
- Branch B at C_a=1.5: reduced Lambda_inf={b15[b15.beta==0].Lambda_inf.iloc[0]:.8e}; beta=50 parent Lambda_inf={b15[b15.beta==50].Lambda_inf.iloc[0]:.8e}.

## Claim boundary

The radial form is derived from the parent-action Hessian, but the numerical Hessian projections are one explicit constitutive closure. They are not yet unique or parameter-free ECSM coefficients. The sign changes are conditional benchmark results, not universal ECSM predictions.

## Contents

- `tables/`: imported-input audit, equations, frozen recovery, coupling scans, stability diagnostics, Branch B recurrence and multipole extraction, claim ledger, and test summary.
- `figures/`: ten benchmark figures.
- `equations/`: copyable LaTeX equations for the parent tidal system and Branch B asymptotic extraction.
- `inputs/`: the imported N30R benchmark and zero-crossing tables.
- `metadata.json`: run metadata, hashes, and test count.
'''
(OUT/'README.md').write_text(readme)

(OUT/'environment.txt').write_text('\n'.join([
    f'python={sys.version}',f'platform={platform.platform()}',f'numpy={np.__version__}',f'pandas={pd.__version__}',
    f'sympy={sp.__version__}',f'mpmath={mp.__version__}'
]))

archive=shutil.make_archive(str(BASE/'ecsm_n31r_outputs'),'zip',root_dir=OUT)
print('Passed:',int(summary_df['pass'].sum()),'/',len(summary_df))
print('Output archive:',archive)

## Conclusion

N31R advances the tidal programme from an anisotropic-fluid benchmark to a parent-field perturbation calculation.

The static parent system is

\[
\boxed{
H''+P_0H'+Q_0H=\frac{c_\chi\delta\chi+c_n\delta n}{\mu},
}
\]

coupled to the coherence and amplitude Hessian equations. The phase perturbation consistently vanishes on the static zero-flux branch with no external phase tide.

The exact frozen limit is

\[
\boxed{\delta\chi=\delta n=\delta\theta=0\quad\Longrightarrow\quad\text{N30R}.}
\]

Within the explicit threshold-gated closure, the N30R negative branch is not universal: parent fields preserve it away from threshold, shift its zero crossing, and reverse the sign in a narrow locally stable compactness interval.

For the noncompact Branch B atmosphere, the response is extracted from an atmosphere-adapted asymptotic basis rather than an arbitrary cutoff. The resonant \(r^{-3}\) coefficient is isolated directly, giving

\[
\boxed{\Lambda_\infty=\frac{B}{3AM^5}.}
\]

The correct N31R claim is therefore

\[
\boxed{
\text{ECSM now has an executable parent-field tidal closure, an exact N30R reduced limit, and a cutoff-free Branch B multipole extraction.}
}
\]

A unique ECSM Love-number prediction still requires the primitive derivation of the parent Hessian coefficients.